In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [3]:
df=pd.read_csv(r"data\superstore_raw.csv",encoding='latin-1')

In [5]:
print(df.shape)          

(9994, 21)


In [13]:
pd.DataFrame({"missing value":df.isnull().sum(),"Data Type":df.dtypes})

,missing value,Data Type
Row ID,0,int64
Order ID,0,str
Order Date,0,datetime64[us]
Ship Date,0,datetime64[us]
Ship Mode,0,str
Customer ID,0,str
Customer Name,0,str
Segment,0,str
Country,0,str
City,0,str


In [9]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [10]:
# Convert date columns from string to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=False)


In [11]:
# Extract useful time columns
df['Year']    = df['Order Date'].dt.year
df['Month']   = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter
df['Month_Name'] = df['Order Date'].dt.strftime('%b')


In [12]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,Month,Quarter,Month_Name
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,4,Nov
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,4,Nov
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,2,Jun
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,4,Oct
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,4,Oct


In [14]:
# Profit Margin % per row
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales'] * 100).round(2)

In [16]:
# Shipping lag in days
df['Ship_Lag_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

In [17]:
# Flag loss-making orders
df['Is_Loss'] = df['Profit'] < 0

In [18]:
# Revenue after discount (cross-check)
df['Discounted_Revenue'] = df['Sales'] * (1 - df['Discount'])

In [19]:
print(df[['Sales','Profit','Profit_Margin_Pct','Ship_Lag_Days']].head())

      Sales    Profit  Profit_Margin_Pct  Ship_Lag_Days
0  261.9600   41.9136              16.00              3
1  731.9400  219.5820              30.00              3
2   14.6200    6.8714              47.00              4
3  957.5775 -383.0310             -40.00              7
4   22.3680    2.5164              11.25              7


In [20]:
# Check for extreme discounts
print(df['Discount'].value_counts().sort_index())

Discount
0.00    4798
0.10      94
0.15      52
0.20    3657
0.30     227
0.32      27
0.40     206
0.45      11
0.50      66
0.60     138
0.70     418
0.80     300
Name: count, dtype: int64


In [22]:
# Check for extremely high/negative profit
print(df['Profit'].describe())
df[df['Profit'] < -500][['Product Name','Sales','Profit','Discount']]

count    9994.000000
mean       28.656896
std       234.260108
min     -6599.978000
25%         1.728750
50%         8.666500
75%        29.364000
max      8399.976000
Name: Profit, dtype: float64


,Product Name,Sales,Profit,Discount
27,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.430,-1665.0522,0.5
165,Lexmark MX611dhe Monochrome Laser Printer,8159.952,-1359.9920,0.4
215,Cisco 9971 IP Video Phone Charcoal,1188.000,-950.4000,0.7
262,Lexmark MX611dhe Monochrome Laser Printer,3059.982,-509.9970,0.4
463,Bush Advantage Collection Racetrack Conference...,1272.630,-814.4832,0.5
683,Cubify CubeX 3D Printer Triple Head Print,7999.980,-3839.9904,0.5
869,GBC Ibimaster 500 Manual ProClick Binding System,1141.470,-760.9800,0.7
949,"Riverside Furniture Oval Coffee Table, Oval En...",2065.320,-619.5960,0.4
1199,GBC DocuBind P400 Electric Binding System,1088.792,-1850.9464,0.8
1369,"Atlantic Metals Mobile 4-Shelf Bookcases, Cust...",590.058,-786.7440,0.7


In [23]:
# Save clean file
df.to_csv('data/superstore_clean.csv', index=False)
print("Clean file saved. Shape:", df.shape)

Clean file saved. Shape: (9994, 29)
